<a href="https://colab.research.google.com/github/inhyuk78/Re-implementation-of-MOLI-By-Hossein-Sharifi-Noghabi-/blob/main/03_Autoencoder_Feature_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
drug_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/drug.parquet')
rnaseq_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/rnaseq_clean.parquet')
mutation_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/mutation_clean.parquet')
CNA_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/CNA_clean.parquet')

In [ ]:
# Each encoding sub-network has a fully connected layer with ReLU
# Additionally, each encoding sub-network employs dropout to regularize model & batch normalization

In [ ]:
print(f'Gene expression data shape: {rnaseq_df.shape}')
print(f'Mutation data shape: {mutation_df.shape}')
print(f'CNA data shape: {CNA_df.shape}')

Gene expression data shape: (944, 36416)
Mutation data shape: (965, 20468)
CNA data shape: (961, 17256)


## **Converting LN_IC50 to binary label in drug response data**

- IC50: represents drug concentration required to inhibit cancer growth by 50%
- LN_IC50: Natural logarithm of IC50
    - [If LN_IC50 < 0]:   0 < IC50 < 1 (high potency - less drug required to inhibit cancer growth by 50%)
    - [If LN_IC50 = 0]:   IC50 = 1 (moderate potency)
    - [If LN_IC50 > 0]:   IC50 > 1 (low potency - more drug required to inhibit cancer growth by 50%)

In [ ]:
# Sensitive (1) if LN_IC50 < 0, Resistant (0) if LN_IC50 > 0
drug_df['label'] = (drug_df['LN_IC50'] < 0).astype(int)
drug_df.head()

,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,LN_IC50,label
0,PFSK-1,SIDM01132,Other Solid Cancers,1003,Camptothecin,TOP1,-1.463887,1
1,A673,SIDM00848,Ewing's Sarcoma,1003,Camptothecin,TOP1,-4.869455,1
2,ES5,SIDM00263,Ewing's Sarcoma,1003,Camptothecin,TOP1,-3.360586,1
3,ES7,SIDM00269,Ewing's Sarcoma,1003,Camptothecin,TOP1,-5.044940,1
4,EW-11,SIDM00203,Ewing's Sarcoma,1003,Camptothecin,TOP1,-3.741991,1


## **Sample alignment**

In [ ]:
x = rnaseq_df.index

common_lines = x.intersection(mutation_df.index).intersection(CNA_df.index)

print(len(common_lines))
print(common_lines)

937
Index(['SIDM00003', 'SIDM00023', 'SIDM00040', 'SIDM00041', 'SIDM00042',
       'SIDM00043', 'SIDM00044', 'SIDM00045', 'SIDM00046', 'SIDM00047',
       ...
       'SIDM01237', 'SIDM01240', 'SIDM01242', 'SIDM01245', 'SIDM01246',
       'SIDM01247', 'SIDM01248', 'SIDM01251', 'SIDM01259', 'SIDM01265'],
      dtype='object', name='model_id', length=937)


In [ ]:
rnaseq_df = rnaseq_df.loc[common_lines]
mutation_df = mutation_df.loc[common_lines]
CNA_df = CNA_df.loc[common_lines]

In [ ]:
drug_df_filtered = drug_df[drug_df['SANGER_MODEL_ID'].isin(common_lines)]

print(len(drug_df_filtered['SANGER_MODEL_ID'].unique()))

937


###**Extracting drug of interest**

- Paper screened with: Docetaxel, Cisplatin, Gemcitabine, Paclitaxel, Erlotinib, Cetuximab
- This project focuses on Gemcitabine because it had the most balanced distribution of sensitive and resistant cell lines in the GDSC dataset (look below)


In [ ]:
group = drug_df[drug_df['DRUG_NAME']=='Gemcitabine'].set_index('SANGER_MODEL_ID')

common = []

for cl in common_lines:
  if cl in group.index:
    common.append(cl)

labels_df = group.loc[common, 'label']

print(len(labels_df))

928


In [ ]:
n_sensitive = labels_df.sum().item()
n_resistant = len(labels_df) - n_sensitive
print(f'Gemcitabine: sensitive: {n_sensitive}, resistant: {n_resistant}')

Gemcitabine: sensitive: 566, resistant: 362


###**Data pre-processing before encoding**

In [ ]:
train_lines, test_lines = train_test_split(
    common,
    test_size = 0.25,
    random_state=64,
    shuffle=True,
    stratify=labels_df.values # Ensure same number of 0/1 in label
)

print(f'Train cell lines: {len(train_lines)}')
print(f'Test cell lines: {len(test_lines)}')

Train cell lines: 696
Test cell lines: 232


In [ ]:
rnaseq_train_df = rnaseq_df.loc[train_lines]
rnaseq_test_df = rnaseq_df.loc[test_lines]

mutation_train_df = mutation_df.loc[train_lines]
mutation_test_df = mutation_df.loc[test_lines]

CNA_train_df = CNA_df.loc[train_lines]
CNA_test_df = CNA_df.loc[test_lines]

y_train = torch.tensor(labels_df.loc[train_lines].values, dtype=torch.float32)
y_test = torch.tensor(labels_df.loc[test_lines].values, dtype=torch.float32)

In [ ]:
scaler = StandardScaler()

# Scaler only applied to rnaseq data as mutation, CNA data values = 0,1
rnaseq_train_scaled = scaler.fit_transform(rnaseq_train_df.values)
rnaseq_test_scaled = scaler.transform(rnaseq_test_df.values)

rnaseq_train_tensor = torch.tensor(rnaseq_train_scaled, dtype=torch.float32)
rnaseq_test_tensor = torch.tensor(rnaseq_test_scaled, dtype=torch.float32)

mutation_train_tensor = torch.tensor(mutation_train_df.values, dtype=torch.float32)
mutation_test_tensor = torch.tensor(mutation_test_df.values, dtype=torch.float32)

CNA_train_tensor = torch.tensor(CNA_train_df.values, dtype=torch.float32)
CNA_test_tensor = torch.tensor(CNA_test_df.values, dtype=torch.float32)

###**Model architecture of Autoencoder**

In [ ]:
class Autoencoder(nn.Module):
  def __init__(self, input_dim, latent_dim):
    super(Autoencoder, self).__init__()

    # Encoder: input_dim -> latent_dim
    self.encoder = nn.Sequential(
        nn.Linear(input_dim, 1024),   # Compress to 1-5% (1024 features) of input dimension
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, latent_dim)
    )

    # Decoder: latent_dim -> input_dim
    self.decoder = nn.Sequential(
        nn.Linear(latent_dim, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(1024, input_dim)
    )

  def forward(self, x):
    encoded = self.encoder(x)
    reconstructed = self.decoder(encoded)
    return encoded, reconstructed

## **Feature extraction using Autoencoder**

In [ ]:
def train_autoencoder(model, dataloader, epochs, learning_rate):
  optimizer = optim.Adam(model.parameters(), lr=learning_rate)
  loss_fn = nn.MSELoss()

  model.train()
  for epoch in range(epochs):
    total_loss = 0
    for batch in dataloader:
      optimizer.zero_grad()
      encoded, reconstructed = model(batch)
      loss = loss_fn(reconstructed, batch)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()

    if epoch % 10 == 0:
      print(f'Epoch: {epoch} | Loss: {total_loss}')

### 1) Preparation before running through Autoencoder

In [ ]:
rnaseq_train_loader = DataLoader(rnaseq_train_tensor, batch_size=64, shuffle=False)
mutation_train_loader = DataLoader(mutation_train_tensor, batch_size=64, shuffle=False)
CNA_train_loader = DataLoader(CNA_train_tensor, batch_size=64, shuffle=False)

### 2) Training each autoencoder separately per dataset

In [ ]:
epochs = 50
lr = 0.01
latent_dim = 256

ae_rnaseq = Autoencoder(input_dim=rnaseq_train_tensor.shape[1], latent_dim=latent_dim)
train_autoencoder(ae_rnaseq, rnaseq_train_loader, epochs=epochs, learning_rate=lr)

ae_mutation = Autoencoder(input_dim=mutation_train_tensor.shape[1], latent_dim=latent_dim)
train_autoencoder(ae_mutation, mutation_train_loader, epochs=epochs, learning_rate=lr)

ae_cna = Autoencoder(input_dim=CNA_train_tensor.shape[1], latent_dim=latent_dim)
train_autoencoder(ae_cna, CNA_train_loader, epochs=epochs, learning_rate=lr)

Epoch: 0 | Loss: 26.701215982437134
Epoch: 10 | Loss: 8.953977823257446
Epoch: 20 | Loss: 8.098113179206848
Epoch: 30 | Loss: 7.5218505859375
Epoch: 40 | Loss: 6.958948969841003
Epoch: 0 | Loss: 10.569661185145378
Epoch: 10 | Loss: 0.09489078633487225
Epoch: 20 | Loss: 0.042867765529081225
Epoch: 30 | Loss: 0.01662898901849985
Epoch: 40 | Loss: 0.008812435844447464
Epoch: 0 | Loss: 21.766802072525024
Epoch: 10 | Loss: 2.6655413210392
Epoch: 20 | Loss: 2.3258777856826782
Epoch: 30 | Loss: 2.0911018550395966
Epoch: 40 | Loss: 1.8338768482208252


### 3) Extracting features using Trained model

In [ ]:
def extract_features(model, tensor, batch_size=64):
  model.eval()

  loader = DataLoader(tensor, batch_size=batch_size, shuffle=False)

  features = []

  with torch.no_grad():
    for batch in loader:
      encoded, _ = model(batch)
      features.append(encoded)

  return torch.cat(features,dim=0)

In [ ]:
# rnaseq data

rnaseq_train_feat = extract_features(ae_rnaseq, rnaseq_train_tensor)
rnaseq_test_feat = extract_features(ae_rnaseq, rnaseq_test_tensor)

In [ ]:
# Mutation data

mutation_train_feat = extract_features(ae_mutation, mutation_train_tensor)
mutation_test_feat = extract_features(ae_mutation, mutation_test_tensor)

In [ ]:
# CNA data

CNA_train_feat = extract_features(ae_cna, CNA_train_tensor)
CNA_test_feat = extract_features(ae_cna, CNA_test_tensor)

## **Concatenating learned features into ONE multi-omics representation**



In [ ]:
X_train = torch.cat([rnaseq_train_feat, mutation_train_feat, CNA_train_feat], dim=1)
X_test = torch.cat([rnaseq_test_feat, mutation_test_feat, CNA_test_feat], dim=1)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

X_train shape: torch.Size([696, 768])
X_test shape: torch.Size([232, 768])
y_train shape: torch.Size([696])
y_test shape: torch.Size([232])


In [ ]:
torch.save({'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test}
           , '/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/multi_omics.pt')